In [1]:
# Se estiveres num ambiente limpo, podes instalar dependências:
# %pip install -q seaborn pyampute

from typing import Optional, List

import numpy as np
import pandas as pd
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from pyampute.ampute import MultivariateAmputation

from missingfcup.core.MissingData import MissingData
from missingfcup.plots.Panel import Panel

In [2]:
_df_raw = sns.load_dataset("titanic")

# Preserve deck for numeric encoding used later
_deck_raw = _df_raw["deck"] if "deck" in _df_raw.columns else None

# Ensure we work with numeric variables only (encode categoricals as numeric codes)
for col in _df_raw.columns:
    if not pd.api.types.is_numeric_dtype(_df_raw[col]):
        codes = pd.Categorical(_df_raw[col]).codes.astype("float64")
        _df_raw[col] = pd.Series(codes, index=_df_raw.index).replace(-1, np.nan)

# Add numeric deck encoding for later plots
if _deck_raw is not None:
    deck_codes = pd.Categorical(_deck_raw).codes.astype("float64")
    _df_raw["deck_num"] = pd.Series(deck_codes, index=_df_raw.index).replace(-1, np.nan)

_df_raw.head()

# Ground truth completo
# Removemos NaNs originais para garantir uma base sem missingness
# df_complete = _df_raw.dropna().reset_index(drop=True)

# Variáveis derivadas (contínuas e intuitivas)
# df_complete["family_size"] = df_complete["sibsp"] + df_complete["parch"] + 1
# df_complete["fare_per_person"] = df_complete["fare"] / df_complete["family_size"]

# Variáveis numéricas observáveis adicionais
# sex_code = 1 (male), 0 (female)
# df_complete["sex_code"] = (df_complete["sex"] == "male").astype(int)

# print("Dimensões do dataset completo:", df_complete.shape)

# Mostramos apenas algumas colunas para contexto
#cols_preview = ["survived", "sex", "age", "pclass", "fare", "sibsp", "parch"]
#df_complete[cols_preview].head()

#df_complete.head()

df = MissingData(_df_raw)


In [3]:
df.column_missing_rate_heatmap().show()

In [4]:
df.missing_count_barchart().show()

In [5]:
df.upset_plot().show()

In [6]:
df.pattern_barchart().show()

In [7]:
df.heatmap().show()

In [8]:
df.missingness_correlation_heatmap().show()

In [9]:
df.dendrogram().show()

In [10]:
# Temporary numeric encoding of deck only for this visualization
deck_num = pd.Series(
    pd.Categorical(df.data["deck"]).codes,
    index=df.data.index,
    dtype="float64",
).replace(-1, np.nan)


In [11]:

df.scatterplot("age", "deck").show()

In [12]:
df.parallel_coordinates(
    #missingness_color_column="deck",
    ).show()

In [13]:
result = df.littles_mcar_test()
print(result)

chi2                                                   702.230589
df                                                             56
p_value                                                       0.0
n_patterns                                                      5
n_rows                                                        891
n_columns                                                      16
columns_used    [survived, pclass, sex, age, sibsp, parch, far...
dtype: object


1. Missing Completely at Random (MCAR).
No MCAR because p-value < 0.05. 

In embarked, and embark_town: Only 2 values are missing here. If you check these two passengers (Mrs. Stone and Miss Icard), they were 1st-class passengers traveling together. There is no strong evidence that their port is missing because of their class or survival. It’s likely a recording error.

In [ ]:
# 1) parallel_coordinates: pclass, fare, survived with binary color for age missingness
md_parallel = df.parallel_coordinates(
    selected_columns=["pclass", "fare", "survived"],
    missingness_color_column="age",
    present_color="#1f77b4",  # Age Present (Blue)
    missing_color="#d62728",  # Age Missing (Red)
    title="Parallel Coordinates: Age Missing vs Present",
)
md_parallel.show()

In [15]:

# 2) scatterplot: Pclass vs Fare, colored by age missingness
md_scatter = df.scatterplot(
    x="pclass",
    y="age",
    missingness_color_column="age",
    title="Pclass vs Fare (Colored by Age Missingness)",
)
md_scatter.show()


## Missingness Mechanisms (Visual Evidence)

### 1. Age — Missing at Random (MAR)

**Mechanism:** The data is missing because of other observed variables (specifically `pclass` and `sex`), not because of the age itself.

**Analysis:** In 1912, record-keeping was more meticulous for First Class passengers. If you look at the data, the `age` column is missing significantly more often for 3rd-class passengers.

**The Why:** It’s not that the passenger's age made them hide it; it’s that the system (Class) determined whether their age was recorded.

### 2. Deck/Cabin — Missing Not at Random (MNAR)

**Mechanism:** The data is missing because of the value itself or the nature of the variable.

**Analysis:** `deck` is derived from the `cabin` number. Most 3rd-class passengers didn't have a cabin; they stayed in large steerage dorms. Therefore, `deck` is missing because they weren't in a cabin.

**The Why:** The missingness is a direct reflection of the passenger's status. It’s **informative missingness**.


In [28]:
# Visual evidence for MAR (age) and MNAR (deck)

# MAR: age missingness is associated with observed variables (pclass, sex)
# 1) Scatterplot with age missingness coloring

df.scatterplot(
    x="fare",
    y="age",
    missingness_color_column="age",
    present_color="#1f77b4",  # Age Present (Blue)
    missing_color="#d62728",  # Age Missing (Red)
    title="MAR: Age Missingness vs Pclass and Fare",
).show()



In [ ]:
# 2) Correlation heatmap for missingness among age, pclass, and sex

df.all_correlation_heatmap().show()

/Users/matias/Library/Python/3.9/lib/python/site-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning:

invalid value encountered in divide

/Users/matias/Library/Python/3.9/lib/python/site-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning:

invalid value encountered in divide



In [26]:

# MNAR: deck missingness reflects the variable itself (cabin/deck not assigned)
# 1) Heatmap ordered by class to show deck missingness concentration

df.heatmap(
    selected_columns=["fare", "age"],
    title="MNAR: Age Missingness by Class",
    order_by=[{"axis": "rows", "column": "fare", "type": "numeric", "ascending": True}],
).show()


In [19]:

# 2) UpSet plot highlighting deck_num

df.upset_plot(
    max_sets=0,
    title="MNAR Signal: Deck Highlight in Missingness Intersections",
    missing_color="#bdbdbd",
    highlight_columns=["deck_num"],
    highlight_color="#d62728",
).show()


## MAR: Parallel + Heatmap (Ordered by Pclass)

We use parallel coordinates (colored by **age missingness**) and a heatmap ordered by `pclass` to highlight that missing `age` is associated with class and other observed variables.


In [20]:
# Parallel coordinates to show association with age missingness

df.parallel_coordinates(
    selected_columns=["pclass", "sex", "fare", "survived", "sibsp", "parch"],
    missingness_color_column="age",
    present_color="#1f77b4",  # Age Present (Blue)
    missing_color="#d62728",  # Age Missing (Red)
    title="MAR: Age Missingness vs Pclass and Other Variables",
).show()

# Heatmap ordered by pclass to reveal concentration of missing age

df.heatmap(
    selected_columns=["age", "pclass", "sex", "fare"],
    title="MAR: Age Missingness Ordered by Pclass",
    order_by=[{"axis": "rows", "column": "pclass", "type": "numeric", "ascending": True}],
    max_columns=10,
).show()
